# US Approval Forecasting — Exploratory Data Analysis

Data flows: `data_utils.EconomicDataFetcher` → `data_loader.DataLoader` → this notebook.

Sections:
1. Setup & data loading
2. Dataset overview
3. Null value analysis
4. FRED economic indicators (time series)
5. Media sentiment
6. President approval ratings

## 1. Setup & data loading

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Load .env from the project root (two levels up from src/)
load_dotenv(Path.cwd().parent / ".env")

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns

# Add the lambda package to sys.path so fetch_sources resolves
sys.path.insert(0, str(Path.cwd().parent / "pipelines" / "lambda"))

from fetch_sources.fred import FREDFetcher
from fetch_sources.gdelt import GDELTFetcher
from fetch_sources.votehub import VoteHubFetcher

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

In [ ]:
import json
from datetime import date

START = "2017-01-01"   # Trump term 1 start; adjust as needed
END   = None           # None = up to latest available

RESOURCES_DIR = Path.cwd() / "resources"
RESOURCES_DIR.mkdir(exist_ok=True)

today = date.today().isoformat()  # e.g. "2026-04-17"

def _resource_path(name: str) -> Path:
    return RESOURCES_DIR / f"{name}_{today}.json"

def _load_or_fetch(name: str, fetch_fn):
    path = _resource_path(name)
    if path.exists():
        print(f"[cache] Loading {name} from {path.name}")
        return json.loads(path.read_text())
    print(f"[fetch] Fetching {name} ...")
    data = fetch_fn()
    path.write_text(json.dumps(data))
    return data

raw_fred      = _load_or_fetch("fred",      lambda: FREDFetcher(api_key=os.getenv("FRED_API_KEY", "")).fetch_panel(start_date=START, end_date=END))
raw_polls     = _load_or_fetch("polls",     lambda: VoteHubFetcher().fetch())
raw_sentiment = _load_or_fetch("sentiment", lambda: GDELTFetcher().fetch())

# --- FRED → wide DataFrame (date index × series columns) ---
if raw_fred:
    panel = (
        pd.DataFrame(raw_fred)
        .assign(date=lambda df: pd.to_datetime(df["date"]))
        .pivot_table(index="date", columns="series", values="value", aggfunc="last")
        .sort_index()
    )
    panel.columns.name = None
else:
    panel = pd.DataFrame()

# --- Polls → long DataFrame ---
if raw_polls:
    polls = (
        pd.DataFrame(raw_polls)
        .assign(date=lambda df: pd.to_datetime(df["date"]))
        .sort_values("date")
        .reset_index(drop=True)
    )
else:
    polls = pd.DataFrame()

# --- Sentiment → long DataFrame ---
if raw_sentiment:
    sentiment = (
        pd.DataFrame(raw_sentiment)
        .assign(date=lambda df: pd.to_datetime(df["date"]))
        .sort_values("date")
        .reset_index(drop=True)
    )
else:
    sentiment = pd.DataFrame()

print(f"economic       : {panel.shape}")
print(f"approval       : {polls.shape}")
print(f"media_sentiment: {sentiment.shape}")

In [ ]:
# ── JSON schema overview (for SQL table design) ───────────────────────────────
# Each source is described from its raw JSON (flat records) — the natural
# shape for a relational table, not the pivoted panel used for analysis.

def _infer_sql_type(col: str, value) -> str:
    if col == "date" or (isinstance(value, str) and len(value) == 10 and value[4] == "-"):
        return "DATE"
    if isinstance(value, int):
        return "INTEGER"
    if isinstance(value, float):
        return "DOUBLE PRECISION"
    return "VARCHAR(255)"

sources = {
    "fred":      raw_fred,
    "polls":     raw_polls,
    "sentiment": raw_sentiment,
}

for table, records in sources.items():
    print(f"\n{'═' * 62}")
    print(f"  TABLE: {table}   ({len(records):,} records)")
    print(f"{'═' * 62}")

    if not records:
        print("  (no data — table will be empty)")
        continue

    sample = records[0]

    # Detect nullable columns by scanning up to 500 records
    nullable = {col: any(r.get(col) is None for r in records[:500]) for col in sample}

    schema_rows = []
    for col, val in sample.items():
        schema_rows.append({
            "column":     col,
            "py_type":    type(val).__name__,
            "sql_type":   _infer_sql_type(col, val),
            "nullable":   nullable[col],
            "example":    val,
        })

    display(pd.DataFrame(schema_rows).set_index("column"))

    # Suggested CREATE TABLE DDL
    col_defs = ",\n    ".join(
        f"{r['column']:<20} {r['sql_type']}{'' if r['nullable'] else ' NOT NULL'}"
        for r in schema_rows
    )
    print(f"\n-- Suggested DDL:\nCREATE TABLE {table} (\n    {col_defs}\n);\n")

## 2. Dataset overview

In [ ]:
panel.head()

In [ ]:
panel.dtypes.to_frame(name="dtype")

In [ ]:
panel.describe().T.round(2)

## 3. Null value analysis

In [ ]:
null_counts = panel.isnull().sum()
null_pct    = (null_counts / len(panel) * 100).round(1)

null_summary = (
    pd.DataFrame({"missing": null_counts, "pct": null_pct})
    .sort_values("missing", ascending=False)
)

print(f"Total rows : {len(panel)}")
print(f"Columns with any null: {(null_counts > 0).sum()} / {len(panel.columns)}")
null_summary[null_summary["missing"] > 0]

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
cols_with_nulls = null_summary[null_summary["missing"] > 0].index.tolist()

if cols_with_nulls:
    sns.heatmap(
        panel[cols_with_nulls].isnull(),
        cbar=False,
        yticklabels=False,
        cmap="Reds",
        ax=ax,
    )
    ax.set_title("Missing value positions (red = null)")
    ax.set_xlabel("Feature")
    ax.set_ylabel("Time (rows)")
    plt.xticks(rotation=45, ha="right", fontsize=8)
else:
    ax.text(0.5, 0.5, "No missing values", ha="center", va="center", fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Forward-fill then back-fill handles leading/trailing nulls from different series
# frequencies (e.g. quarterly GDP producing NaN months).
panel_clean = panel.ffill().bfill()

remaining_nulls = panel_clean.isnull().sum().sum()
print(f"Nulls remaining after ffill/bfill: {remaining_nulls}")

# Drop columns that are entirely null (series the API returned empty)
panel_clean = panel_clean.dropna(axis=1, how="all")
print(f"Panel shape after cleaning: {panel_clean.shape}")

## 4. FRED economic indicators (time series)

In [ ]:
numeric_cols = panel_clean.select_dtypes(include="number").columns.tolist()
print(f"Numeric features: {len(numeric_cols)}")
panel_clean[numeric_cols].describe().T[["mean", "std", "min", "max"]].round(3).head()

In [ ]:
import matplotlib.dates as mdates

n_cols  = len(numeric_cols)
n_grid  = 4
n_rows  = int(np.ceil(n_cols / n_grid))

fig, axes = plt.subplots(n_rows, n_grid, figsize=(n_grid * 4, n_rows * 3))
axes_flat = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes_flat[i]
    series = panel_clean[col].dropna()
    ax.plot(series.index, series.values, linewidth=1.0, color="steelblue")
    ax.set_title(col, fontsize=8, pad=3)
    ax.set_ylabel("value", fontsize=7)
    ax.tick_params(labelsize=6)
    locator = mdates.AutoDateLocator(maxticks=4)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle("FRED Economic Indicators (Raw Time Series)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Skewness & kurtosis summary — useful for spotting non-normal series
dist_stats = pd.DataFrame({
    "skewness": panel_clean[numeric_cols].skew(),
    "kurtosis": panel_clean[numeric_cols].kurtosis(),
}).sort_values("skewness", key=abs, ascending=False)

print("Top features by absolute skewness:")
dist_stats.round(3)

## 5. Media sentiment

In [ ]:
if sentiment.empty:
    print("No sentiment data available.")
else:
    fig, ax1 = plt.subplots(figsize=(14, 5))

    color_tone   = "steelblue"
    color_volume = "coral"

    ax1.plot(sentiment["date"], sentiment["tone"], color=color_tone, linewidth=1.5, label="Tone")
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Tone (rolling avg)", color=color_tone)
    ax1.tick_params(axis="y", labelcolor=color_tone)

    ax2 = ax1.twinx()
    ax2.fill_between(sentiment["date"], sentiment["volume"], alpha=0.25, color=color_volume, label="Volume")
    ax2.set_ylabel("Article volume (rolling avg)", color=color_volume)
    ax2.tick_params(axis="y", labelcolor=color_volume)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    ax1.set_title("Media Sentiment (GDELT): Tone & Article Volume")
    plt.tight_layout()
    plt.show()

## 6. President approval ratings

In [ ]:
if polls.empty:
    print("No approval data available.")
else:
    fig, ax = plt.subplots(figsize=(14, 5))

    if "approval" in polls.columns:
        ax.scatter(polls["date"], polls["approval"], s=20, alpha=0.5, color="steelblue", label="Approval")
        rolling_approve = polls.set_index("date")["approval"].rolling("30D").mean()
        ax.plot(rolling_approve.index, rolling_approve.values, color="steelblue", linewidth=2)

    if "disapproval" in polls.columns:
        ax.scatter(polls["date"], polls["disapproval"], s=20, alpha=0.5, color="coral", label="Disapproval")
        rolling_disapprove = polls.set_index("date")["disapproval"].rolling("30D").mean()
        ax.plot(rolling_disapprove.index, rolling_disapprove.values, color="coral", linewidth=2)

    ax.set_xlabel("Date")
    ax.set_ylabel("Percentage (%)")
    ax.set_title("President Approval Ratings (dots = individual polls, line = 30-day rolling mean)")
    ax.legend()
    plt.tight_layout()
    plt.show()